First Agent

In [1]:
from google.adk.agents.llm_agent import Agent

- description = human-readable summary of the agent’s purpose.(no effect on model)
- instruction = sustem message / system prompt

In [2]:
root_agent = Agent(
    model='gemini-2.5-flash',
    name='first_agent',
    description='A helpful assistant for user questions.',
    instruction='Answer user questions to the best of your knowledge',
)

Agent + TOOL

In [3]:
def greeting_tool() -> str:
    """Returns a warm, friendly greeting."""
    return "Hello from your specialized greeting tool! Welcome."

In [4]:
agent2 = Agent(
    model='gemini-2.5-flash',
    name='second_agent',
    description='A friendly agent that provides a special greeting.',
    instruction='You are a friendly agent. When the user greets you, you MUST use the greeting_tool to respond.',
    tools=[greeting_tool],
)

## Components of an ADK Agent

### LLM response generation : controls how the underlying large language model produces its responses

The generate_content_config parameter, which lets you adjust settings like temperature (randomness), max_output_tokens (response length), and safety filters to shape the output behavior

In [5]:
from google.adk.agents.llm_agent import LlmAgent
from google.genai import types

root_agent = LlmAgent(
    name='greeting_agent',
    model='gemini-2.5-flash',
    description='An agent that provides a friendly greeting in a specified language.',
    instruction='You are a friendly agent. Greet the user in their specified language.',
    generate_content_config=types.GenerateContentConfig(
        temperature=0.2,
        max_output_tokens=250,
        safety_settings=[
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                threshold=types.HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
            )
        ]
    )
)

### Structured Data/Data Validation

In [6]:
from google.adk.agents.llm_agent import LlmAgent
from pydantic import BaseModel, Field

In [7]:
# Define schemas for structured input and output
class GreetingRequest(BaseModel):
    """Input schema for specifying the language of the greeting."""
    language: str = Field(description="The language to greet the user in.")

class GreetingResponse(BaseModel):
    """Output schema for the structured greeting response."""
    greeting_message: str = Field(description="The final, formatted greeting.")

In [8]:
root_agent = LlmAgent(
    name='greeting_agent',
    model='gemini-2.5-flash',
    description='A helpful assistant for user questions.',
    instruction='Answer user questions to the best of your knowledge',
    input_schema=GreetingRequest,
    output_schema=GreetingResponse,
    output_key='final_greeting',
)

### Context Management

 include_contents: parameter to control whether the agent receives the prior conversation history

In [9]:
root_agent = LlmAgent(
    name='greeting_agent',
    model='gemini-2.5-flash',
    description='An agent that provides a friendly greeting in a specified language.',
    instruction='You are a friendly agent. Greet the user in their specified language.',
    include_contents='none',
)

By default, the include_contents parameter is automatically set to 'default'. This means that unless explicitly configured otherwise, the ADK will always provide the relevant conversation history to the agent

### Planning

The planner parameter in an LlmAgent enables multi-step reasoning and planning before execution

In [10]:
from google.adk.agents.llm_agent import LlmAgent
from google.adk.planners import BuiltInPlanner
from google.genai import types

- BuiltInPlanner: It leverages the model’s native thinking or planning capabilities 
    - thinking_budget: It controls the number of thinking tokens when generating a response
    - include_thoughts: It controls whether the model returns its internal reasoning process along with the final answer.

In [11]:
root_agent = LlmAgent(
    name='greeting_agent',
    model='gemini-2.5-flash',
    description='An agent that provides a friendly greeting in a specified language.',
    instruction='You are a friendly agent. Greet the user in their specified language.',
    planner=BuiltInPlanner(
        thinking_config=types.ThinkingConfig(
            include_thoughts=True,
            thinking_budget=1024,
        )
    ),
)

- PlanReActPlanner: It instructs the model to follow a specific Plan -> Action -> Reason structure, which is useful for models that do not have a built-in thinking feature.

In [12]:
from google.adk.planners import PlanReActPlanner

In [13]:
root_agent = LlmAgent(
    name='greeting_agent',
    model='gemini-2.0-flash',
    description='An agent that provides a friendly greeting in a specified language.',
    instruction='You are a friendly agent. Greet the user in their specified language.',
    planner=PlanReActPlanner(),
)

### Code execution

Allows an agent to execute blocks of code (like Python) generated as part of its response

In [14]:
from google.adk.code_executors import BuiltInCodeExecutor

In [15]:
root_agent = LlmAgent(
    name='greeting_agent',
    model='gemini-2.5-flash',
    description='An agent that provides a friendly greeting in a specified language.',
    instruction='You are a friendly agent. Greet the user in their specified language.',
    code_executor=BuiltInCodeExecutor(),
)

we have given the agent the ability to perform tasks like calculations, data manipulation, or running small scripts

## Tools

### Custom Tools

Docstring: This is the most critical piece. The function’s docstring is used as the description of the tool 
- A detailed docstring that clearly explains what the tool does, what each parameter means, and what it returns is essential for the LLM to understand when and how to use the tool correctly.`

In [16]:
def create_greeting(name: str, language: str = "English") -> str:
    """Creates a personalized greeting for a user in a specified language.

    Args:
        name (str): The name of the person to greet.
        language (str): The language for the greeting. Defaults to English.
    """
    if language.lower() == "spanish":
        return f"Hola, {name}! Cómo estás?"
    else:
        return f"Hello, {name}! How are you?"

In [17]:
root_agent = LlmAgent(
    name='greeting_tool_agent',
    model='gemini-2.5-flash',
    instruction="""You are a helpful greeter. When the user asks for a greeting,
    use the `create_greeting` tool to generate it.""",
    tools=[create_greeting], # The framework automatically wraps this function as a tool
)

### Built-in tools

In [18]:
from google.adk.tools import google_search

* use the bypass_multi_tools_limit=True parameter to combine both built inan custom tools

In [19]:
root_agent = LlmAgent(
    name='basic_search_agent',
    model='gemini-2.5-flash',
    instruction="Answer user questions by searching the internet.",
    tools=[google_search],
)

- google_search tool : allows an agent to perform real-time Google searches
- BuiltInCodeExecutor: It allows an agent to run generated code in a secure environment.
- VertexAiSearchTool and VertexAiRagRetrieval: They enable search across private, configured data stores and documents. 
- BigQuery: It is a set of tools for asking questions about data in BigQuery tables using natural language.
- Spanner: It is a set of tools for interacting with and querying Spanner databases.

### Agents-as-tools

In [20]:
from google.adk.agents.llm_agent import LlmAgent
from google.adk.tools.agent_tool import AgentTool

In [21]:
# Define the specialized worker agent that will be used as a tool.
greeting_expert = LlmAgent(
    name='greeting_expert_agent',
    model='gemini-2.5-flash',
    description='This agent is an expert at creating personalized greetings in different languages.',
    instruction="""You are a greeting expert. A user will provide a name and a language.
    Create a friendly, personalized greeting. For example: Hola, Maria!"""
)

In [22]:
root_agent = Agent(
    name='delegator_agent',
    model='gemini-2.5-flash',
    instruction="""You are a helpful assistant. If the user asks for any kind of greeting,
    delegate the task to the `greeting_expert_agent` tool. Forward the user's
    request exactly as you receive it.""",
    
    # Wrap the worker agent in AgentTool and provide it as a tool to the manager.
    tools=[AgentTool(agent=greeting_expert)]
)

## Execution Engine = The Runner

 Manages the event loop 
 - it receives user input, runs the agent’s logic until an event (like a tool call or final answer) is yielded, processes that event (e.g., executes a tool), then resumes the agent with updated information

In [24]:
import asyncio
import os

from google.adk.agents.llm_agent import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

In [23]:
chat_agent = LlmAgent(
    model="gemini-2.5-flash", 
    name="chat_agent",
    description="A friendly assistant for conversations.",
    instruction=(
        "You are a helpful chat assistant. "
        "Answer the user's questions clearly."
    ),
)

In [26]:
async def main() -> None:
    if not os.getenv("GOOGLE_API_KEY"):
        raise RuntimeError("GOOGLE_API_KEY environment variable is not set")

    # Create an in-memory runner for this agent
    runner = InMemoryRunner(
        agent=chat_agent,
        app_name="chat_app",
    )

    # Create a session (kept in memory only for this run)
    session = await runner.session_service.create_session(
        app_name=runner.app_name,
        user_id="local-user", 
    )

    print("ADK Chat Agent is ready.")
    print("Type your message, or 'exit' / 'quit' to stop.\n")

    # Simple chat loop
    while True:
        try:
            user_text = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nExiting chat.")
            break

        if user_text.lower() in {"exit", "quit"}:
            print("Goodbye!")
            break

        if not user_text:
            continue

        # Build the user message as a Content object
        new_message = genai_types.Content(
            role="user",
            parts=[genai_types.Part(text=user_text)],
        )

        # Collect the model's textual reply
        reply_chunks: list[str] = []

        # Run the agent through the runner for this turn
        async for event in runner.run_async(
            user_id=session.user_id,
            session_id=session.id,
            new_message=new_message,
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    text = getattr(part, "text", None)
                    if text:
                        reply_chunks.append(text)

        # Print the last assembled reply (if any)
        if reply_chunks:
            # Join all text parts for this turn
            reply_text = "".join(reply_chunks)
            print(f"Agent: {reply_text}\n")
        else:
            print("Agent: (no response content received)\n")


In [ ]:
if __name__ == "__main__":
    asyncio.run(main())